# Collection Broken Links vs. UMM-C `RelatedUrls`

For every collection in the consolidated lineage evidence files, this notebook takes the `broken_links` field
and checks each broken link. Is it one of the collection's own UMM-C `RelatedUrls`? Then it lives in CMR
metadata and the DAAC needs to fix it. If not, the link was found somewhere else during lineage research
(an ATBD, a landing page, a user guide, a README, ...).

This repeats `Notebooks/Broken_Links_RelatedUrls_CrossCheck.ipynb` with two changes:

* The input is now `lineage/metrics/<PROVIDER>/*_lineage_evidence.json`. Each file is a group-level consolidation
  (`consolidated` header plus a `collections` list) instead of one file per collection.
* `RelatedUrls` come **live from CMR GraphQL**, not from the deleted `eosdis_collections_w_metadata_associations.json`
  snapshot. The full `ummMetadata` is fetched as well. A broken link that isn't in `RelatedUrls` but is somewhere else
  in the UMM-C record (citation `OnlineResource`, data-center contact URLs, `DOI`, ...) is still a CMR metadata problem,
  so it gets its own category.
* Because the comparison is against the *current* record, a link the DAAC has fixed since the research would look
  like a non-CMR link. So for any link that isn't in the current record, the UMM-C revision that was current at research time
  is fetched from CMR (`all_revisions=true`) and checked as well.

**Input**
- `lineage/metrics/<PROVIDER>/*_lineage_evidence.json` for `ASF`, `GES_DISC`, `GHRC_DAAC`, `LANCEMODIS`, `LARC_CLOUD`,
  `LPCLOUD`, `LPDAAC_ECS`
- CMR GraphQL: `https://graphql.earthdata.nasa.gov/api`

**Output**
- `lineage/reports/Collection_BrokenLinks_Report.md`

**Classification of each broken link**

| Category | Meaning | Who fixes it |
|---|---|---|
| `RelatedUrls` | The URL is one of the collection's UMM-C `RelatedUrls` (exact or normalized match) | DAAC, in the CMR record |
| `Other UMM-C field` | Not in `RelatedUrls`, but it appears somewhere else in the collection's UMM-C record | DAAC, in the CMR record |
| `Fixed in CMR since research` | Not in the current record, but it was in the UMM-C revision that was current when the lineage research ran. The DAAC has already removed or replaced it | Nobody. Already done |
| `Lineage research` | Not anywhere in the current CMR record. It was found while researching lineage (ATBD, document, landing page, ...) | Document or site owner. No CMR change needed |
| `Collection not in CMR` | The concept ID no longer resolves in CMR, so the link can't be classified | n/a |

**Steps**
1. Inventory the consolidated lineage evidence files and the `broken_links` field
2. Fetch the current UMM-C records from CMR GraphQL
3. Define the matching rules
4. Classify every broken link (and check the research-time CMR revision for links missing from the current record)
5. Analysis
6. Supplement: other link checks recorded in the evidence files
7. Summary
8. Write the report

In [1]:
import json
import re
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlsplit

import requests

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lineage" / "metrics").is_dir())
METRICS_DIR = REPO_ROOT / "lineage" / "metrics"
REPORT_MD = REPO_ROOT / "lineage" / "reports" / "Collection_BrokenLinks_Report.md"
PROVIDERS = ["ASF", "GES_DISC", "GHRC_DAAC", "LANCEMODIS", "LARC_CLOUD", "LPCLOUD", "LPDAAC_ECS"]
GRAPHQL_URL = "https://graphql.earthdata.nasa.gov/api"
BATCH_SIZE = 100   # concept IDs per GraphQL request

session = requests.Session()
print(f"Repo root: {REPO_ROOT}")
for prov in PROVIDERS:
    print(f"  {prov:<11} exists={(METRICS_DIR / prov).is_dir()}")

Repo root: /Users/eleu/dev/esdis/metadata-quality-skills
  ASF         exists=True
  GES_DISC    exists=True
  GHRC_DAAC   exists=True
  LANCEMODIS  exists=True
  LARC_CLOUD  exists=True
  LPCLOUD     exists=True
  LPDAAC_ECS  exists=True


## Step 1: Inventory the lineage evidence files and the `broken_links` field

Each `*_lineage_evidence.json` holds one collection group. `collections[*]` is the original per-collection evidence
record. The target block is not uniform across records: the short name is under either `short_name` or `shortname`,
but `concept_id` is always there, so records are matched to CMR by concept ID.

`broken_links` is a `{url: http_status_or_error}` dict. A record can lack the field entirely (no link check was
recorded), have an empty dict (links were checked and none were broken), or have a non-empty dict.

In [2]:
records = []   # one per collection
for prov in PROVIDERS:
    for path in sorted((METRICS_DIR / prov).glob("*_lineage_evidence.json")):
        doc = json.loads(path.read_text())
        group = doc["consolidated"]["group"]
        for ev in doc["collections"]:
            tgt = ev.get("target", {})
            broken = ev.get("broken_links")
            records.append({
                "provider": prov,
                "group": group,
                "file": path.name,
                "short_name": tgt.get("short_name") or tgt.get("shortname"),
                "version": str(tgt.get("version") or ""),
                "concept_id": tgt.get("concept_id"),
                "revision_at_research": tgt.get("revision_id"),
                "researched_at": ev.get("generated") or ev.get("generated_at") or doc["consolidated"].get("generated"),
                "broken_state": "absent" if broken is None else ("non-empty" if broken else "empty"),
                "broken_links": broken or {},
                "evidence": ev,
            })

print(f"Evidence files: {len({(r['provider'], r['file']) for r in records})}, collections: {len(records)}")
print(f"Collections without concept_id: {sum(not r['concept_id'] for r in records)}")
print(f"Duplicate concept_ids: {[c for c, n in Counter(r['concept_id'] for r in records).items() if n > 1] or 'none'}\n")

print(f"{'Provider':<11} {'files':>5} {'collections':>11} {'absent':>7} {'empty':>6} {'non-empty':>9} {'broken links':>12}")
for prov in PROVIDERS:
    rs = [r for r in records if r["provider"] == prov]
    st = Counter(r["broken_state"] for r in rs)
    print(f"{prov:<11} {len({r['file'] for r in rs}):>5} {len(rs):>11} {st['absent']:>7} {st['empty']:>6} "
          f"{st['non-empty']:>9} {sum(len(r['broken_links']) for r in rs):>12}")
print(f"{'ALL':<11} {len({(r['provider'], r['file']) for r in records}):>5} {len(records):>11} "
      f"{sum(r['broken_state'] == 'absent' for r in records):>7} {sum(r['broken_state'] == 'empty' for r in records):>6} "
      f"{sum(r['broken_state'] == 'non-empty' for r in records):>9} {sum(len(r['broken_links']) for r in records):>12}")

Evidence files: 51, collections: 747
Collections without concept_id: 0
Duplicate concept_ids: none

Provider    files collections  absent  empty non-empty broken links
ASF             6          41      41      0         0            0
GES_DISC        9         157     157      0         0            0
GHRC_DAAC       1           1       1      0         0            0
LANCEMODIS      4          20      20      0         0            0
LARC_CLOUD     29         523     248      7       268          579
LPCLOUD         1           3       3      0         0            0
LPDAAC_ECS      1           2       2      0         0            0
ALL            51         747     472      7       268          579


In [3]:
# Which LARC_CLOUD groups have the field, and which don't
by_group = defaultdict(Counter)
for r in records:
    by_group[(r["provider"], r["group"])][r["broken_state"]] += 1
print("Groups that have a broken_links field in at least one record:")
for (prov, grp), st in sorted(by_group.items()):
    if st["empty"] or st["non-empty"]:
        print(f"  {prov:<11} {grp:<16} non-empty={st['non-empty']:<4} empty={st['empty']:<3} absent={st['absent']}")
print("\nStatus values seen in broken_links:",
      dict(Counter(str(s) for r in records for s in r["broken_links"].values()).most_common()))

Groups that have a broken_links field in at least one record:
  LARC_CLOUD  CALIPSO          non-empty=34   empty=1   absent=0
  LARC_CLOUD  CATS             non-empty=16   empty=0   absent=0
  LARC_CLOUD  CLAMS            non-empty=3    empty=0   absent=0
  LARC_CLOUD  DSCOVR           non-empty=18   empty=1   absent=0
  LARC_CLOUD  ERBE             non-empty=8    empty=0   absent=0
  LARC_CLOUD  FLASHFlux        non-empty=1    empty=0   absent=0
  LARC_CLOUD  GloSSAC          non-empty=7    empty=1   absent=0
  LARC_CLOUD  MISR_ARCTAS      non-empty=6    empty=0   absent=0
  LARC_CLOUD  MISR_CMARE       non-empty=4    empty=0   absent=0
  LARC_CLOUD  MISR_GoMACCS     non-empty=6    empty=0   absent=0
  LARC_CLOUD  MISR_ICARTT      non-empty=2    empty=0   absent=0
  LARC_CLOUD  MISR_INTEXB      non-empty=6    empty=0   absent=0
  LARC_CLOUD  PREFIRE          non-empty=34   empty=0   absent=0
  LARC_CLOUD  SAGE_II          non-empty=1    empty=0   absent=0
  LARC_CLOUD  SAGE_III      

## Step 2: Fetch the current UMM-C records from CMR GraphQL

Only collections with a non-empty `broken_links` need classifying, but every collection is fetched. That way the
report can confirm that each concept ID still resolves, and Step 6 can reuse the records.

Concept IDs are sent in batches of 100 (`CollectionsInput.conceptId` takes a list). Only these fields are requested:
`conceptId revisionId revisionDate shortName version provider relatedUrls ummMetadata`. Failed requests are retried with
exponential backoff, and the paging loop stops on the item count, the same way as in
`Collection_MetadataAssociations_CrossCheck.ipynb`.

In [4]:
QUERY = """
query Collections($params: CollectionsInput) {
  collections(params: $params) {
    count
    cursor
    items { conceptId revisionId revisionDate shortName version provider relatedUrls ummMetadata }
  }
}
"""


def post_graphql(params, retries=5):
    """Return the `collections` block. Retries with exponential backoff and raises if every attempt fails."""
    err = None
    for attempt in range(retries):
        try:
            r = session.post(GRAPHQL_URL, timeout=180, json={"query": QUERY, "variables": {"params": params}})
            body = r.json()
            data = (body.get("data") or {}).get("collections")
            if r.status_code == 200 and data is not None:
                return data
            err = f"HTTP {r.status_code}: {(body.get('errors') or [{}])[0].get('message')}"
        except (requests.RequestException, ValueError) as e:
            err = repr(e)
        time.sleep(2 ** attempt)
    raise RuntimeError(f"GraphQL failed after {retries} attempts: {err}")


def fetch_all(params, page_size=BATCH_SIZE):
    items, p = [], {**params, "limit": page_size}
    while True:
        data = post_graphql(p)
        items.extend(data["items"] or [])
        if len(items) >= data["count"] or not data["items"]:
            return items
        p = {**params, "limit": page_size, "cursor": data["cursor"]}


t0 = time.time()
concept_ids = sorted({r["concept_id"] for r in records if r["concept_id"]})
cmr = {}
for i in range(0, len(concept_ids), BATCH_SIZE):
    for item in fetch_all({"conceptId": concept_ids[i:i + BATCH_SIZE]}):
        cmr[item["conceptId"]] = item
retrieved_at = datetime.now(timezone.utc).isoformat(timespec="seconds")

missing = [cid for cid in concept_ids if cid not in cmr]
print(f"Requested {len(concept_ids)} concept IDs in {-(-len(concept_ids) // BATCH_SIZE)} batches, "
      f"got {len(cmr)} records in {time.time() - t0:.1f} s")
print(f"Retrieved at (UTC): {retrieved_at}")
print(f"Concept IDs not found in CMR: {missing or 'none'}")

changed = [r for r in records if r["concept_id"] in cmr and r["revision_at_research"] is not None
           and str(r["revision_at_research"]) != str(cmr[r["concept_id"]]["revisionId"])]
print(f"Records whose CMR revision changed since the lineage research: {len(changed)} "
      f"(of {sum(r['revision_at_research'] is not None for r in records)} with a recorded revision_id)")

Requested 747 concept IDs in 8 batches, got 747 records in 11.9 s
Retrieved at (UTC): 2026-11-25T17:31:53+00:00
Concept IDs not found in CMR: none
Records whose CMR revision changed since the lineage research: 90 (of 388 with a recorded revision_id)


## Step 3: Define the matching rules

**URL normalization.** Two URLs count as the same link if they are equal after: trimming whitespace, lowercasing,
dropping the scheme (`http`/`https`), dropping a leading `www.`, dropping the `#fragment`, and dropping a trailing `/`.
Every match is labeled `exact` (identical strings) or `normalized`, so a loose match is visible in the report.

**Where a URL can live in UMM-C.** `RelatedUrls[*].URL` is checked first. If there's no match there, every other
string value in `ummMetadata` that looks like a URL is checked, and its JSON path is recorded (for example
`CollectionCitations[0].OnlineResource.Linkage`). A bare DOI in `DOI.DOI` (`10.5067/...`) also counts as
`https://doi.org/<DOI>`.

**Research-time revision.** For a link that isn't in the current record, the collection's revision history is fetched
from the CMR search API (`collections.umm_json?concept_id=...&all_revisions=true`). The research-time revision is the
`revision_id` recorded in the evidence `target`. If none was recorded, it's the latest revision whose `revision-date`
is on or before the evidence record's `generated` timestamp. If the link is in that revision, it is classified
`Fixed in CMR since research`.

**Where a non-CMR link came from.** For links not in CMR, the collection's own evidence record is searched for the
URL (`source_documents`, `fetch_log`, `atbd_search`, ...). The first place it appears, plus the
`source_documents` title and `evidence_source` where available, says what kind of research turned it up.

In [5]:
def normalize(url):
    u = (url or "").strip().lower()
    u = re.sub(r"^[a-z][a-z0-9+.-]*://", "", u)
    u = re.sub(r"^www\.", "", u)
    u = u.split("#", 1)[0]
    return u.rstrip("/")


URL_RE = re.compile(r"^(https?|ftp)://", re.I)


def walk(obj, path=""):
    """Yield (json_path, string_value) for every string in a nested JSON value."""
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield from walk(v, f"{path}.{k}" if path else k)
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from walk(v, f"{path}[{i}]")
    elif isinstance(obj, str):
        yield path, obj


def umm_url_index(item):
    """Return (related, other). related: list of RelatedUrls dicts; other: {normalized_url: (path, url)}."""
    related = []
    for ru in item.get("relatedUrls") or []:
        url = ru.get("url") or ru.get("uRL") or ru.get("URL")
        if url:
            related.append({**ru, "url": url})
    other = {}
    for path, val in walk(item.get("ummMetadata") or {}):
        if path.startswith("RelatedUrls"):
            continue
        v = val.strip()
        if URL_RE.match(v):
            other.setdefault(normalize(v), (path, v))
        elif path.endswith("DOI") and v.startswith("10."):
            other.setdefault(normalize("https://doi.org/" + v), (path, "https://doi.org/" + v))
    return related, other


def research_context(ev, url):
    """Where in the evidence record (outside broken_links) the URL appears, plus document details if available."""
    n = normalize(url)
    for doc in ev.get("source_documents") or []:
        if isinstance(doc, dict) and normalize(doc.get("url")) == n:
            return "source_documents", doc.get("title"), doc.get("evidence_source")
    rest = {k: v for k, v in ev.items() if k != "broken_links"}
    for path, val in walk(rest):
        if normalize(val) == n:
            return path, None, None
    for path, val in walk(rest):   # URL quoted inside free text (e.g. metadata_findings)
        if url.strip() in val:
            return path, None, None
    return None, None, None


CMR_SEARCH = "https://cmr.earthdata.nasa.gov/search/collections.umm_json"
_history = {}


def research_revision(r):
    """Return (revision_id, revision_date, item-like dict) for the CMR revision that was current at research time."""
    cid = r["concept_id"]
    if cid not in _history:
        for attempt in range(5):
            try:
                resp = session.get(CMR_SEARCH, timeout=120,
                                   params={"concept_id": cid, "all_revisions": "true", "page_size": 2000})
                resp.raise_for_status()
                _history[cid] = resp.json()["items"]
                break
            except (requests.RequestException, ValueError):
                time.sleep(2 ** attempt)
        else:
            raise RuntimeError(f"revision history fetch failed for {cid}")
    revs = [i for i in _history[cid] if not i["meta"].get("deleted")]
    if r["revision_at_research"] is not None:
        hit = [i for i in revs if str(i["meta"]["revision-id"]) == str(r["revision_at_research"])]
    else:
        hit = sorted((i for i in revs if r["researched_at"] and i["meta"]["revision-date"] <= r["researched_at"]),
                     key=lambda i: i["meta"]["revision-id"])[-1:]
    if not hit:
        return None, None, None
    umm = hit[0]["umm"]
    item = {"relatedUrls": [{"url": ru.get("URL"), "type": ru.get("Type"), "subtype": ru.get("Subtype")}
                            for ru in umm.get("RelatedUrls") or []], "ummMetadata": umm}
    return hit[0]["meta"]["revision-id"], hit[0]["meta"]["revision-date"], item


def rtype(ru):
    return " / ".join(x for x in (ru.get("type"), ru.get("subtype")) if x)


# Self-test of the normalizer
assert normalize("http://www.Example.com/a/#x") == normalize("https://example.com/a")
assert normalize("https://example.com/a") != normalize("https://example.com/a/b")
print("normalize() self-test passed")

normalize() self-test passed


## Step 4: Classify every broken link

In [6]:
links = []   # one row per (collection, broken link)
for r in records:
    if not r["broken_links"]:
        continue
    item = cmr.get(r["concept_id"])
    related, other = umm_url_index(item) if item else ([], {})
    exact = {ru["url"].strip(): ru for ru in related}
    norm = {normalize(ru["url"]): ru for ru in related}
    for url, status in r["broken_links"].items():
        row = {**{k: r[k] for k in ("provider", "group", "short_name", "version", "concept_id")},
               "url": url, "status": str(status), "category": None, "match": None,
               "related_type": None, "cmr_url": None, "umm_path": None,
               "found_in": None, "doc_title": None, "evidence_source": None,
               "research_rev": None, "current_rev": None}
        if item is None:
            row["category"] = "Collection not in CMR"
        elif url.strip() in exact:
            ru = exact[url.strip()]
            row.update(category="RelatedUrls", match="exact", related_type=rtype(ru), cmr_url=ru["url"])
        elif normalize(url) in norm:
            ru = norm[normalize(url)]
            row.update(category="RelatedUrls", match="normalized", related_type=rtype(ru), cmr_url=ru["url"])
        elif normalize(url) in other:
            path, val = other[normalize(url)]
            row.update(category="Other UMM-C field", match="exact" if val == url.strip() else "normalized",
                       umm_path=path, cmr_url=val)
        else:
            row["category"] = "Lineage research"
            rev_id, rev_date, old = research_revision(r)
            if old is not None and str(rev_id) != str(item["revisionId"]):
                o_related, o_other = umm_url_index(old)
                o_norm = {normalize(ru["url"]): ru for ru in o_related}
                if normalize(url) in o_norm or normalize(url) in o_other:
                    ru = o_norm.get(normalize(url))
                    row.update(category="Fixed in CMR since research",
                               related_type=rtype(ru) if ru else None,
                               umm_path=None if ru else o_other[normalize(url)][0],
                               cmr_url=ru["url"] if ru else o_other[normalize(url)][1],
                               match="exact" if (ru["url"] if ru else o_other[normalize(url)][1]) == url.strip() else "normalized",
                               research_rev=f"{rev_id} ({rev_date[:10]})", current_rev=f"{item['revisionId']} ({item.get('revisionDate', '')[:10]})")
        if row["category"] not in ("RelatedUrls", "Fixed in CMR since research"):
            row["found_in"], row["doc_title"], row["evidence_source"] = research_context(r["evidence"], url)
        links.append(row)

CATEGORIES = ["RelatedUrls", "Other UMM-C field", "Fixed in CMR since research", "Lineage research", "Collection not in CMR"]
print(f"Broken links classified: {len(links)} across {len({l['concept_id'] for l in links})} collections\n")
print("By category:  ", {c: sum(l["category"] == c for l in links) for c in CATEGORIES})
print("By match type:", dict(Counter((l["category"], l["match"]) for l in links)))
print(f"Revision histories fetched: {len(_history)}")

Broken links classified: 579 across 268 collections

By category:   {'RelatedUrls': 556, 'Other UMM-C field': 0, 'Fixed in CMR since research': 23, 'Lineage research': 0, 'Collection not in CMR': 0}
By match type: {('RelatedUrls', 'exact'): 556, ('Fixed in CMR since research', 'exact'): 23}
Revision histories fetched: 4


## Step 5: Analysis

In [7]:
HDR = ("Data center", "Collections checked (field present)", "Collections w/ broken links", "Broken links",
       "In RelatedUrls", "Other UMM-C field", "Fixed in CMR since research", "Lineage research",
       "Collection not in CMR", "Collections needing a CMR fix")
summary_rows = []
for prov in PROVIDERS:
    rs = [r for r in records if r["provider"] == prov]
    ls = [l for l in links if l["provider"] == prov]
    summary_rows.append((
        prov,
        sum(r["broken_state"] != "absent" for r in rs),
        sum(r["broken_state"] == "non-empty" for r in rs),
        len(ls),
        *(sum(l["category"] == c for l in ls) for c in CATEGORIES),
        len({l["concept_id"] for l in ls if l["category"] in ("RelatedUrls", "Other UMM-C field")}),
    ))
totals = tuple(sum(row[i] for row in summary_rows) for i in range(1, len(HDR)))

print(" | ".join(HDR))
for row in summary_rows:
    print(" | ".join(map(str, row)))
print("TOTAL | " + " | ".join(map(str, totals)))

Data center | Collections checked (field present) | Collections w/ broken links | Broken links | In RelatedUrls | Other UMM-C field | Fixed in CMR since research | Lineage research | Collection not in CMR | Collections needing a CMR fix
ASF | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0
GES_DISC | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0
GHRC_DAAC | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0
LANCEMODIS | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0
LARC_CLOUD | 275 | 268 | 579 | 556 | 0 | 23 | 0 | 0 | 264
LPCLOUD | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0
LPDAAC_ECS | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0
TOTAL | 275 | 268 | 579 | 556 | 0 | 23 | 0 | 0 | 264


In [8]:
print("HTTP status / error of broken links, by category:")
for c in CATEGORIES:
    st = Counter(l["status"] for l in links if l["category"] == c)
    if st:
        print(f"  {c:<22} {dict(st.most_common())}")

print("\nRelatedUrls Type / Subtype of the broken links that are in RelatedUrls:")
for t, n in Counter(l["related_type"] for l in links if l["category"] == "RelatedUrls").most_common():
    print(f"  {n:4d}  {t}")

print("\nBroken links by host:")
for h, n in Counter(urlsplit(l["url"]).netloc.lower() for l in links).most_common(15):
    print(f"  {n:4d}  {h}")

HTTP status / error of broken links, by category:
  RelatedUrls            {'404': 525, '403': 24, 'SSLError': 7}
  Fixed in CMR since research {'404': 13, 'ConnectTimeout': 7, 'ConnectionError': 3}

RelatedUrls Type / Subtype of the broken links that are in RelatedUrls:
   237  VIEW RELATED INFORMATION / DATA CITATION GUIDELINES
    70  VIEW RELATED INFORMATION / GENERAL DOCUMENTATION
    65  PROJECT HOME PAGE
    38  DATA SET LANDING PAGE
    24  VIEW RELATED INFORMATION / MICRO ARTICLE
    24  HOME PAGE
    21  VIEW RELATED INFORMATION / PRODUCTION HISTORY
    15  VIEW RELATED INFORMATION / USER'S GUIDE
    12  VIEW RELATED INFORMATION / PROCESSING HISTORY
    11  VIEW RELATED INFORMATION / USER FEEDBACK PAGE
    10  VIEW RELATED INFORMATION / PUBLICATIONS
     9  VIEW RELATED INFORMATION / HOW-TO
     7  PROFESSIONAL HOME PAGE
     6  VIEW RELATED INFORMATION / DATA QUALITY
     5  VIEW RELATED INFORMATION / PRODUCT USAGE
     2  VIEW RELATED INFORMATION / ALGORITHM THEORETICAL BAS

### Distinct broken URLs

Most of the broken links are a handful of URLs shared by many collections. Fixing each one at the source (or adding a
redirect) clears every collection that uses it. The table below lists distinct URLs by how many collections they
affect.

In [9]:
distinct = defaultdict(lambda: {"collections": set(), "groups": set(), "status": Counter(), "cat": Counter(), "type": Counter()})
for l in links:
    d = distinct[l["url"]]
    d["collections"].add(l["concept_id"]); d["groups"].add(l["group"])
    d["status"][l["status"]] += 1; d["cat"][l["category"]] += 1
    if l["related_type"]:
        d["type"][l["related_type"]] += 1
distinct_sorted = sorted(distinct.items(), key=lambda kv: (-len(kv[1]["collections"]), kv[0]))
print(f"Distinct broken URLs: {len(distinct)}")
n_top = 10
top_cover = sum(len(d["collections"]) for _, d in distinct_sorted[:n_top])
print(f"The top {n_top} URLs account for {top_cover} of {len(links)} broken-link occurrences "
      f"({top_cover / max(len(links), 1):.0%})\n")
for url, d in distinct_sorted[:25]:
    print(f"{len(d['collections']):4d}  {url}  [{', '.join(d['type']) or ', '.join(d['cat'])}] "
          f"status={dict(d['status'])}")

Distinct broken URLs: 58
The top 10 URLs account for 434 of 579 broken-link occurrences (75%)

 248  https://asdc.larc.nasa.gov/citing-data  [VIEW RELATED INFORMATION / DATA CITATION GUIDELINES, VIEW RELATED INFORMATION / GENERAL DOCUMENTATION] status={'404': 248}
  34  https://asdc.larc.nasa.gov/project/CALIPSO  [VIEW RELATED INFORMATION / GENERAL DOCUMENTATION, PROJECT HOME PAGE] status={'404': 34}
  34  https://asdc.larc.nasa.gov/project/PREFIRE  [DATA SET LANDING PAGE] status={'404': 34}
  24  https://asdc.larc.nasa.gov/project/MISR  [HOME PAGE] status={'404': 24}
  24  https://www.jpl.nasa.gov/spaceimages/details.php?id=PIA21927  [VIEW RELATED INFORMATION / MICRO ARTICLE] status={'403': 24}
  17  https://asdc.larc.nasa.gov/project/DSCOVR  [PROJECT HOME PAGE] status={'404': 17}
  16  https://asdc.larc.nasa.gov/project/CATS-ISS  [VIEW RELATED INFORMATION / GENERAL DOCUMENTATION] status={'404': 16}
  14  https://www-calipso.larc.nasa.gov/resources/calipso_users_guide/  [VIEW RELATED 

In [10]:
non_related = [l for l in links if l["category"] != "RelatedUrls"]
print(f"Broken links NOT in RelatedUrls: {len(non_related)}")
for l in non_related:
    where = l["research_rev"] and f"rev {l['research_rev']} -> {l['current_rev']}" or l["umm_path"] or l["found_in"] or "?"
    print(f"  {l['short_name']:<32} {l['category']:<28} {where:<46} {l['url']}")

Broken links NOT in RelatedUrls: 23
  CLAMS_CERES_CHESLIGHT_SONDE      Fixed in CMR since research  rev 9 (2026-10-09) -> 10 (2026-11-24)          https://asdc.larc.nasa.gov/project/CLAMS
  CLAMS_CERES_CHESLIGHT_SONDE      Fixed in CMR since research  rev 9 (2026-10-09) -> 10 (2026-11-24)          https://asdc.larc.nasa.gov/citing-data
  CLAMS_CERES_CHESLIGHT_SONDE      Fixed in CMR since research  rev 9 (2026-10-09) -> 10 (2026-11-24)          https://mas.arc.nasa.gov/data/deploy_html/CLAMS_home.html
  CLAMS_CERES_CHESLIGHT_SONDE      Fixed in CMR since research  rev 9 (2026-10-09) -> 10 (2026-11-24)          https://car.gsfc.nasa.gov/
  CLAMS_CERES_CHESLIGHT_SONDE      Fixed in CMR since research  rev 9 (2026-10-09) -> 10 (2026-11-24)          https://modis.gsfc.nasa.gov/
  CLAMS_ER2_MAS                    Fixed in CMR since research  rev 8 (2026-10-09) -> 9 (2026-11-24)           https://asdc.larc.nasa.gov/project/CLAMS
  CLAMS_ER2_MAS                    Fixed in CMR since research 

## Step 6: Supplement – other link checks recorded in the evidence files

`broken_links` exists only in some LARC_CLOUD records. Other records carry their link checks in differently named
fields: `link_check`, `eosweb_link_check`, `citation_link_check`, `readme_link_check`, and `fetch_log`
(document fetches with a `FAILED` status). These failures are **not** part of the `broken_links` analysis above.
They are classified here with the same rules so that no DAAC-actionable link is missed.

An entry counts as a failure when its `status` isn't `ok`/`ok (...)` and isn't an HTTP 2xx/3xx code. URLs that are
already in the same collection's `broken_links` are skipped.

In [11]:
CHECK_FIELDS = ["link_check", "eosweb_link_check", "citation_link_check", "readme_link_check", "fetch_log"]


def is_failure(status):
    if isinstance(status, int):
        return not 200 <= status < 400
    s = str(status).strip().lower()
    if s.isdigit():
        return not 200 <= int(s) < 400
    return not s.startswith("ok")


def check_entries(value):
    """Yield (url, status) from a link-check field (list of dicts, dict of dicts, or a single dict)."""
    if isinstance(value, dict) and "url" in value:
        value = [value]
    elif isinstance(value, dict):
        value = list(value.values())
    for e in value or []:
        if isinstance(e, dict) and e.get("url") and "status" in e:
            status = str(e["status"]).split(" for url:")[0]   # fetch_log repeats the URL in the message
            yield e["url"], status if e.get("error") is None else f"{status} ({e['error']})"


supp = []
for r in records:
    item = cmr.get(r["concept_id"])
    related, other = umm_url_index(item) if item else ([], {})
    norm = {normalize(ru["url"]): ru for ru in related}
    already = {normalize(u) for u in r["broken_links"]}
    seen = set()
    for field in CHECK_FIELDS:
        for url, status in check_entries(r["evidence"].get(field)):
            n = normalize(url)
            if n in already or n in seen or not is_failure(str(status).split(" (")[0]):
                continue
            seen.add(n)
            if item is None:
                cat, where = "Collection not in CMR", None
            elif n in norm:
                cat, where = "RelatedUrls", rtype(norm[n])
            elif n in other:
                cat, where = "Other UMM-C field", other[n][0]
            else:
                cat, where = "Lineage research", None
                rev_id, rev_date, old = research_revision(r)
                if old is not None and str(rev_id) != str(item["revisionId"]):
                    o_related, o_other = umm_url_index(old)
                    if n in {normalize(ru["url"]) for ru in o_related} or n in o_other:
                        cat, where = "Fixed in CMR since research", f"revision {rev_id} -> {item['revisionId']}"
            supp.append({**{k: r[k] for k in ("provider", "group", "short_name", "version", "concept_id")},
                         "field": field, "url": url, "status": str(status), "category": cat, "where": where})

print(f"Additional failed link checks (not in broken_links): {len(supp)} across "
      f"{len({s['concept_id'] for s in supp})} collections\n")
print(f"{'Provider':<11} {'field':<20} " + " ".join(f"{c:>22}" for c in CATEGORIES))
for prov in PROVIDERS:
    for field in CHECK_FIELDS:
        ss = [s for s in supp if s["provider"] == prov and s["field"] == field]
        if ss:
            print(f"{prov:<11} {field:<20} " + " ".join(f"{sum(s['category'] == c for s in ss):>22}" for c in CATEGORIES))
print("\nStatus values:", dict(Counter(s["status"] for s in supp).most_common()))

lc = [r for r in records if r["evidence"].get("link_check") is not None]
print(f"\nRecords with a link_check field: {len(lc)} "
      f"({sum(r['broken_state'] == 'absent' for r in lc)} of them have no broken_links field); "
      f"groups: {sorted({r['group'] for r in lc})}")
supp_cmr = [s for s in supp if s["category"] in ("RelatedUrls", "Other UMM-C field")]
print(f"Supplementary failures still in CMR metadata: {len(supp_cmr)} links in "
      f"{len({s['concept_id'] for s in supp_cmr})} collections")

supp_distinct = defaultdict(lambda: {"collections": set(), "status": Counter(), "cat": Counter(), "fields": set()})
for x in supp:
    d = supp_distinct[x["url"]]
    d["collections"].add(x["concept_id"]); d["status"][x["status"]] += 1
    d["cat"][x["category"]] += 1; d["fields"].add(x["field"])
supp_distinct_sorted = sorted(supp_distinct.items(), key=lambda kv: (-len(kv[1]["collections"]), kv[0]))
print(f"Distinct URLs: {len(supp_distinct)}\n")
for url, d in supp_distinct_sorted[:20]:
    print(f"{len(d['collections']):4d}  {url}  {dict(d['cat'])}  {dict(d['status'])}")

Additional failed link checks (not in broken_links): 649 across 270 collections

Provider    field                           RelatedUrls      Other UMM-C field Fixed in CMR since research       Lineage research  Collection not in CMR
ASF         fetch_log                                 2                      0                      0                      0                      0
GES_DISC    fetch_log                                 0                      0                      0                      3                      0
LARC_CLOUD  link_check                              594                      1                      1                      0                      0
LARC_CLOUD  eosweb_link_check                         5                      1                      3                      0                      0
LARC_CLOUD  citation_link_check                       0                      2                      0                      0                      0
LARC_CLOUD  readme_link_ch

In [12]:
print("Supplementary failures NOT in the current CMR record:")
for s in sorted((s for s in supp if s["category"] not in ("RelatedUrls", "Other UMM-C field")),
                key=lambda s: (s["provider"], s["category"], s["short_name"] or "")):
    print(f"  {s['provider']:<10} {s['short_name'] + ' v' + s['version']:<34} {s['field']:<18} "
          f"{s['category']:<28} {s['status']:<24} {s['url']}")

Supplementary failures NOT in the current CMR record:
  GES_DISC   Global_Landslide_Exposure_Maps v1.0 fetch_log          Lineage research             FAILED: 403 Client Error: Forbidden https://agupubs.onlinelibrary.wiley.com/doi/pdfdirect/10.1002/2017EF000715
  GES_DISC   Global_Landslide_Nowcast v1.1      fetch_log          Lineage research             FAILED: 403 Client Error: Forbidden https://agupubs.onlinelibrary.wiley.com/doi/pdfdirect/10.1002/2017EF000715
  GES_DISC   Global_Landslide_Nowcast v2.0.0    fetch_log          Lineage research             FAILED: 403 Client Error: Forbidden https://agupubs.onlinelibrary.wiley.com/doi/pdfdirect/10.1002/2017EF000715
  LARC_CLOUD ARESE_ER2_MAS v1                   eosweb_link_check  Fixed in CMR since research  unreachable (URLError)   https://eosweb.larc.nasa.gov/sites/default/files/project/arese/guide/arese_project.pdf
  LARC_CLOUD ARESE_ER2_MAS v1                   eosweb_link_check  Fixed in CMR since research  unreachable (URLErro

## Step 7: Summary

**Scope.** 747 collections in 51 consolidated `*_lineage_evidence.json` files across the 7 data centers. Every concept ID
still resolves in CMR (live GraphQL fetch, see the Step 2 timestamp).

**Key findings**

* **Only LARC_CLOUD has `broken_links` results.** The field is in 275 of the 523 LARC_CLOUD records (268 non-empty,
  7 empty). No ASF, GES_DISC, GHRC_DAAC, LANCEMODIS, LPCLOUD, or LPDAAC_ECS record has it, and neither do the other
  248 LARC_CLOUD records. For those collections, "no broken links" means *not checked*, not *clean*.
* **268 collections have 579 broken links. Every one of them was a UMM-C `RelatedUrls` entry when the lineage
  research ran.**
  * **556 links (96%) in 264 collections are still in the current `RelatedUrls`.** All are exact string matches. These
    are CMR metadata problems the DAAC needs to fix.
  * **23 links (4%) in 4 collections are already fixed.** These are CLAMS_CERES_CHESLIGHT_SONDE, CLAMS_ER2_MAS,
    CLAMS_MODIS_L2_AEROSOL_PRODUCTS, and SCAR_A_ER2_MAS. LARC_CLOUD published new revisions on 2026-11-24 that
    replaced them (for example with `earthdata.nasa.gov/data/projects/...` and `asapdata.arc.nasa.gov/emas/...` pages).
    Checked against the live record alone, they would have looked like links found during research. The revision-history
    check shows they were `RelatedUrls`.
  * **0 broken links were found only during lineage research** (ATBDs, documents, landing pages), and 0 were in another
    UMM-C field. The lineage research picked up these broken links while validating each collection's own
    `RelatedUrls`.
* **The breakage comes from a few shared URLs.** There are 58 distinct broken URLs, and the top 10 make up 75% of the
  occurrences. `https://asdc.larc.nasa.gov/citing-data` (DATA CITATION GUIDELINES) alone is broken in 248 collections.
  Next come the ASDC `/project/<MISSION>` pages (CALIPSO, PREFIRE, MISR, DSCOVR, CATS-ISS, TEMPO, GloSSAC, ...) and the
  retired `www-calipso.larc.nasa.gov` user's-guide pages. One redirect or bulk replacement at ASDC for each of these
  URLs would clear most of the findings.
* **HTTP outcomes** for links still in CMR: 525 × 404, 24 × 403 (all the JPL `spaceimages` PIA21927 micro-article,
  likely bot blocking), and 7 × SSLError (`www.sparc-ssirc.org`). Check the 403 and SSL cases by hand before filing a fix.

**Supplement: other link checks (Step 6).** 239 LARC_CLOUD records (CERES, MISR and MISR campaigns, MOPITT, ATOST)
have no `broken_links` but do have a `link_check` field. Counting it, plus `eosweb_link_check`, `citation_link_check`,
`readme_link_check`, and failed `fetch_log` entries, there are **649 more failed links in 270 collections**.
**605 of them (232 collections) are still in CMR metadata**, mostly the same ASDC URLs (`citing-data`,
`/project/MISR`, `/project/CERES`, `/tools-and-services`) and JPL PIA21927. That roughly doubles the DAAC fix list.
The only clearly non-CMR failure is the TEMPO ATBD PDF `asdc.larc.nasa.gov/documents/tempo/ATBD_TEMPO_CH2O.pdf`
(404, found in 32 TEMPO collections). The rest are a few unreachable legacy `eosweb`/`www-misr` pages and paywalled
journal PDFs.

**Caveats.** Link statuses come from the lineage research run, not from this notebook. Matching is done against CMR as of
the Step 2 timestamp, so rerunning the notebook shows DAAC fixes as they land (they move to `Fixed in CMR since research`).

## Step 8: Write the report

The report is built from this run's results. The layout is: overview tables, then each data center, then each
collection, with the broken links numbered under each collection. Collections are sorted by short name and version.

In [13]:
# Summary text from Step 7, written after reviewing this run's output
SUMMARY_LINES = [
    '## Summary',
    '',
    '**Scope.** 747 collections in 51 consolidated `*_lineage_evidence.json` files across the 7 data centers. Every concept ID',
    'still resolves in CMR (live GraphQL fetch, see the Step 2 timestamp).',
    '',
    '**Key findings**',
    '',
    '* **Only LARC_CLOUD has `broken_links` results.** The field is in 275 of the 523 LARC_CLOUD records (268 non-empty,',
    '  7 empty). No ASF, GES_DISC, GHRC_DAAC, LANCEMODIS, LPCLOUD, or LPDAAC_ECS record has it, and neither do the other',
    '  248 LARC_CLOUD records. For those collections, "no broken links" means *not checked*, not *clean*.',
    '* **268 collections have 579 broken links. Every one of them was a UMM-C `RelatedUrls` entry when the lineage',
    '  research ran.**',
    '  * **556 links (96%) in 264 collections are still in the current `RelatedUrls`.** All are exact string matches. These',
    '    are CMR metadata problems the DAAC needs to fix.',
    '  * **23 links (4%) in 4 collections are already fixed.** These are CLAMS_CERES_CHESLIGHT_SONDE, CLAMS_ER2_MAS,',
    '    CLAMS_MODIS_L2_AEROSOL_PRODUCTS, and SCAR_A_ER2_MAS. LARC_CLOUD published new revisions on 2026-11-24 that',
    '    replaced them (for example with `earthdata.nasa.gov/data/projects/...` and `asapdata.arc.nasa.gov/emas/...` pages).',
    '    Checked against the live record alone, they would have looked like links found during research. The revision-history',
    '    check shows they were `RelatedUrls`.',
    '  * **0 broken links were found only during lineage research** (ATBDs, documents, landing pages), and 0 were in another',
    "    UMM-C field. The lineage research picked up these broken links while validating each collection's own",
    '    `RelatedUrls`.',
    '* **The breakage comes from a few shared URLs.** There are 58 distinct broken URLs, and the top 10 make up 75% of the',
    '  occurrences. `https://asdc.larc.nasa.gov/citing-data` (DATA CITATION GUIDELINES) alone is broken in 248 collections.',
    '  Next come the ASDC `/project/<MISSION>` pages (CALIPSO, PREFIRE, MISR, DSCOVR, CATS-ISS, TEMPO, GloSSAC, ...) and the',
    "  retired `www-calipso.larc.nasa.gov` user's-guide pages. One redirect or bulk replacement at ASDC for each of these",
    '  URLs would clear most of the findings.',
    '* **HTTP outcomes** for links still in CMR: 525 × 404, 24 × 403 (all the JPL `spaceimages` PIA21927 micro-article,',
    '  likely bot blocking), and 7 × SSLError (`www.sparc-ssirc.org`). Check the 403 and SSL cases by hand before filing a fix.',
    '',
    '**Supplement: other link checks (Step 6).** 239 LARC_CLOUD records (CERES, MISR and MISR campaigns, MOPITT, ATOST)',
    'have no `broken_links` but do have a `link_check` field. Counting it, plus `eosweb_link_check`, `citation_link_check`,',
    '`readme_link_check`, and failed `fetch_log` entries, there are **649 more failed links in 270 collections**.',
    '**605 of them (232 collections) are still in CMR metadata**, mostly the same ASDC URLs (`citing-data`,',
    '`/project/MISR`, `/project/CERES`, `/tools-and-services`) and JPL PIA21927. That roughly doubles the DAAC fix list.',
    'The only clearly non-CMR failure is the TEMPO ATBD PDF `asdc.larc.nasa.gov/documents/tempo/ATBD_TEMPO_CH2O.pdf`',
    '(404, found in 32 TEMPO collections). The rest are a few unreachable legacy `eosweb`/`www-misr` pages and paywalled',
    'journal PDFs.',
    '',
    '**Caveats.** Link statuses come from the lineage research run, not from this notebook. Matching is done against CMR as of',
    'the Step 2 timestamp, so rerunning the notebook shows DAAC fixes as they land (they move to `Fixed in CMR since research`).',
    '',
]

def md_escape(s):
    return str(s).replace("|", "\\|")


def fmt_link(l):
    if l["category"] == "RelatedUrls":
        tag = f"**RelatedUrls** ({l['related_type'] or 'no Type'}"
        tag += f"; listed as `{l['cmr_url']}`)" if l["match"] == "normalized" else ")"
    elif l["category"] == "Other UMM-C field":
        tag = f"**Other UMM-C field** (`{l['umm_path']}`)"
    elif l["category"] == "Fixed in CMR since research":
        tag = (f"**Fixed in CMR since research** (was {l['related_type'] or l['umm_path']} in revision "
               f"{l['research_rev']}; gone in revision {l['current_rev']})")
    elif l["category"] == "Lineage research":
        src = l["doc_title"] and f"{l['doc_title']}" + (f", {l['evidence_source']}" if l["evidence_source"] else "")
        tag = "Lineage research" + (f" ({src})" if src else (f" (seen in `{l['found_in']}`)" if l["found_in"] else ""))
    else:
        tag = "Collection not in CMR"
    return f"{l['url']} — HTTP {l['status']} — {tag}"


out = [
    "# Collection Broken Links vs. UMM-C RelatedUrls",
    "",
    f"Generated by `lineage/notebooks/Collection_BrokenLinks.ipynb`. CMR UMM-C records were retrieved live from "
    f"CMR GraphQL at **{retrieved_at}** UTC.",
    "",
    "**Source:** the `broken_links` field of every collection in "
    "`lineage/metrics/<PROVIDER>/*_lineage_evidence.json` (providers: " + ", ".join(PROVIDERS) + "). Each broken "
    "link is compared with the collection's current UMM-C `RelatedUrls`, then with every other URL in its UMM-C record.",
    "",
    "**Categories**",
    "",
    "- **RelatedUrls**: the URL is one of the collection's UMM-C `RelatedUrls`. It lives in CMR metadata and needs fixing by the DAAC.",
    "- **Other UMM-C field**: the URL is elsewhere in the UMM-C record (citation, data-center contact, DOI, ...). It also needs fixing by the DAAC.",
    "- **Fixed in CMR since research**: the URL was in the UMM-C revision that was current at research time, but the DAAC has since removed or replaced it. No action needed.",
    "- **Lineage research**: the URL is not in the current CMR record or the research-time one. It was found during lineage research (ATBD, landing page, document, ...).",
    "- **Collection not in CMR**: the concept ID no longer resolves in CMR.",
    "",
    "Matching is exact, or normalized (case-insensitive, ignoring `http`/`https`, leading `www.`, trailing `/`, `#fragment`).",
    "",
    *SUMMARY_LINES,
    "",
    "## Overview",
    "",
    "| " + " | ".join(HDR) + " |",
    "|" + "---|" * len(HDR),
    *("| " + " | ".join(map(str, row)) + " |" for row in summary_rows),
    "| **TOTAL** | " + " | ".join(f"**{t}**" for t in totals) + " |",
    "",
    f"- Collections inventoried: {len(records)} in "
    f"{len({(r['provider'], r['file']) for r in records})} consolidated evidence files.",
    f"- Collections with a `broken_links` field: {sum(r['broken_state'] != 'absent' for r in records)} "
    f"({sum(r['broken_state'] == 'non-empty' for r in records)} non-empty, "
    f"{sum(r['broken_state'] == 'empty' for r in records)} empty). The other "
    f"{sum(r['broken_state'] == 'absent' for r in records)} records have no `broken_links` field, so for them "
    "\"no broken links\" means *not checked*, not *clean*.",
    f"- Concept IDs not found in CMR: {len(missing)}. Records whose CMR revision changed since the research: {len(changed)}.",
    f"- Match type for RelatedUrls hits: "
    + ", ".join(f"{k} = {v}" for k, v in Counter(l["match"] for l in links if l["category"] == "RelatedUrls").items()) + ".",
    "- HTTP status / error: "
    + ", ".join(f"{k} × {v}" for k, v in Counter(l["status"] for l in links).most_common()) + ".",
    "",
    "## Most frequent broken URLs",
    "",
    f"There are {len(distinct)} distinct broken URLs. The top {n_top} account for {top_cover} of "
    f"{len(links)} occurrences ({top_cover / max(len(links), 1):.0%}).",
    "",
    "| # | Collections | URL | RelatedUrls Type / Subtype | Status |",
    "|---|---|---|---|---|",
]
for i, (url, d) in enumerate(distinct_sorted[:25], 1):
    out.append(f"| {i} | {len(d['collections'])} | {md_escape(url)} | "
               f"{md_escape(', '.join(d['type']) or ', '.join(d['cat']))} | "
               f"{', '.join(f'{k}' for k in d['status'])} |")
out.append("")

out += ["## Broken links by data center", ""]
for prov in PROVIDERS:
    out += [f"### {prov}", ""]
    rs = [r for r in records if r["provider"] == prov]
    with_links = sorted({l["concept_id"] for l in links if l["provider"] == prov})
    if not with_links:
        if all(r["broken_state"] == "absent" for r in rs):
            out += [f"_No `broken_links` field in any of the {len(rs)} lineage evidence records for this data center "
                    "(link check not recorded). See the supplement below for other link checks._", ""]
        else:
            out += ["_No broken links found._", ""]
        continue
    recs = sorted((r for r in rs if r["concept_id"] in with_links), key=lambda r: (r["short_name"] or "", r["version"]))
    for r in recs:
        ls = [l for l in links if l["concept_id"] == r["concept_id"]]
        n_cmr = sum(l["category"] in ("RelatedUrls", "Other UMM-C field") for l in ls)
        n_fixed = sum(l["category"] == "Fixed in CMR since research" for l in ls)
        out.append(f"* **{r['short_name']}** (v{r['version']}, {r['concept_id']}, group {r['group']}) — "
                   f"{len(ls)} broken, {n_cmr} still in CMR metadata" + (f", {n_fixed} already fixed" if n_fixed else ""))
        out.append("")
        out += [f"    {i}. {fmt_link(l)}" for i, l in enumerate(ls, 1)]
        out.append("")

out += [
    "## Supplement: other failed link checks (not in `broken_links`)",
    "",
    "These come from `" + "`, `".join(CHECK_FIELDS) + "` in the evidence records. Only failures "
    "(status not `ok` and not HTTP 2xx/3xx) are listed, and URLs already in the collection's `broken_links` are left out. "
    "They are classified with the same rules, including the research-time revision check. `unreachable (URLError)` and "
    "connection errors come from the research environment, so re-check them before filing a fix.",
    "",
    f"- {len(lc)} LARC_CLOUD records have a `link_check` field ({sum(r['broken_state'] == 'absent' for r in lc)} of them "
    f"have no `broken_links`). Groups: {', '.join(sorted({r['group'] for r in lc}))}. For these collections, "
    "`link_check` takes the place of `broken_links`.",
    f"- Additional failed links: **{len(supp)}** in {len({s['concept_id'] for s in supp})} collections. "
    f"**{len(supp_cmr)}** of them, in {len({s['concept_id'] for s in supp_cmr})} collections, are still in the CMR record and "
    "need fixing by the DAAC.",
    "",
    "| Field | " + " | ".join(CATEGORIES) + " |",
    "|---|" + "---|" * len(CATEGORIES),
    *(f"| `{f}` | " + " | ".join(str(sum(s["field"] == f and s["category"] == c for s in supp)) for c in CATEGORIES) + " |"
      for f in CHECK_FIELDS),
    "",
]
if supp:
    out += ["### Most frequent supplementary failures", "",
            "| # | Collections | URL | Category | Status |", "|---|---|---|---|---|"]
    for i, (url, d) in enumerate(supp_distinct_sorted[:20], 1):
        out.append(f"| {i} | {len(d['collections'])} | {md_escape(url)} | {', '.join(d['cat'])} | "
                   f"{md_escape(', '.join(d['status']))} |")
    out += ["", "### Supplementary failures not in the current CMR record", "",
            "| Data center | Collection | Field | Status | Category | URL |", "|---|---|---|---|---|---|"]
    for s in sorted((s for s in supp if s["category"] not in ("RelatedUrls", "Other UMM-C field")),
                    key=lambda s: (s["provider"], CATEGORIES.index(s["category"]), s["short_name"] or "")):
        out.append(f"| {s['provider']} | {md_escape(s['short_name'])} v{s['version']} | {s['field']} | "
                   f"{md_escape(s['status'])} | {s['category']} | {md_escape(s['url'])} |")
    out += ["", "<details><summary>All supplementary failures still in CMR metadata, by collection "
            f"({len(supp_cmr)} links)</summary>", "",
            "| Data center | Collection | Concept ID | Field | Status | Category | Where | URL |",
            "|---|---|---|---|---|---|---|---|"]
    for s in sorted(supp_cmr, key=lambda s: (s["provider"], s["short_name"] or "", s["version"], s["url"])):
        out.append(f"| {s['provider']} | {md_escape(s['short_name'])} v{s['version']} | {s['concept_id']} | {s['field']} | "
                   f"{md_escape(s['status'])} | {s['category']} | {md_escape(s['where'] or '')} | {md_escape(s['url'])} |")
    out += ["", "</details>", ""]

REPORT_MD.write_text("\n".join(out) + "\n")
print(f"Wrote {REPORT_MD.relative_to(REPO_ROOT)} ({len(out)} lines)")

Wrote lineage/reports/Collection_BrokenLinks_Report.md (2222 lines)


In [14]:
print("\n".join(out[:60]))

# Collection Broken Links vs. UMM-C RelatedUrls

Generated by `lineage/notebooks/Collection_BrokenLinks.ipynb`. CMR UMM-C records were retrieved live from CMR GraphQL at **2026-11-25T17:31:53+00:00** UTC.

**Source:** the `broken_links` field of every collection in `lineage/metrics/<PROVIDER>/*_lineage_evidence.json` (providers: ASF, GES_DISC, GHRC_DAAC, LANCEMODIS, LARC_CLOUD, LPCLOUD, LPDAAC_ECS). Each broken link is compared with the collection's current UMM-C `RelatedUrls`, then with every other URL in its UMM-C record.

**Categories**

- **RelatedUrls**: the URL is one of the collection's UMM-C `RelatedUrls`. It lives in CMR metadata and needs fixing by the DAAC.
- **Other UMM-C field**: the URL is elsewhere in the UMM-C record (citation, data-center contact, DOI, ...). It also needs fixing by the DAAC.
- **Fixed in CMR since research**: the URL was in the UMM-C revision that was current at research time, but the DAAC has since removed or replaced it. No action needed.
- **Lineage